# BP1 Gate 5 — Decision Layer & Reporting
**Customer360 Navigator Enterprise Suite — Customer Intent Classification**

## Purpose
Implements Master Execution Plan Section 8 Gate 5: "Decision / GenAI Layer & Reporting — Grounded GenAI
output / decision-engine score with reason codes." Exit criteria: "Every claim carries a citation/evidence
field; every score has reason codes."

## Scope decision (user-confirmed 2026-09-22) — offline decision-record layer, no GenAI API call
The Master Execution Plan's own sprint schedule (Section 24, Sprint 5) and BP list (Section 5/7) place the
actual grounded-GenAI-text and decision-engine work specifically at **BP6 (GenAI Resolution Assistant)** and
**BP7 (Customer Navigator Decision Engine)** — BP1 is a pure intent classifier with no GenAI or cross-BP
decision output of its own. Rather than bolt an early, throwaway GenAI API call onto BP1 (real billing, a
credential to manage, network dependency, and the UDAAP-language-review / PII-screening compliance burden the
Master Plan attaches specifically to "any GenAI call"), this gate satisfies Gate 5's exit criteria for BP1
**without generating any GenAI text at all**:
- **"decision-engine score"** → the champion classifier's own predicted intent + calibrated confidence
  (`predict_proba`), computed live for every held-out test row.
- **"reason codes"** → per-instance SHAP feature attributions (real, computed by this notebook's own run — not
  Gate 4's already-saved *global* top-10, which is a separate, independently cross-checked computation here).
- **"citation/evidence field"** → each reason-code term is only ever reported for a row if that term's TF-IDF
  weight in that row's own document is provably nonzero — i.e. the term is *definitionally* present in that
  complaint's real text, by construction, not asserted after the fact. This is verified as its own structural
  integrity check below, not just assumed.

This keeps BP1 fully offline, deterministic, and reproducible bit-for-bit (no external API, no billing, no
GenAI-hallucination risk) while still producing every field Gate 5's exit criteria name. The compliance
touchpoints Gate 5 names in the Master Plan (UDAAP language review, NIST AI RMF Measure/Manage) are recorded
below as explicitly **Not Applicable to BP1** — stated honestly, the same way the Plan itself handles ECOA when
no protected-class field is present — rather than silently skipped. Real GenAI-call work, with its own PII
screening and UDAAP review, is deferred to BP6 where the Master Plan already places it.

## What this gate computes, concretely
1. Reloads the Gate 3/4 champion (`logistic_regression`), refit live on the full train split — nothing carried
   over from a prior session or notebook.
2. Predicts on the **full held-out test set** (all rows, not a sample) — predicted intent, confidence
   (top-1 `predict_proba`), and the top-3 alternative intents with their probabilities per row.
3. Cross-checks the recomputed overall test accuracy against Gate 3's already-recorded value
   (`gate3_champion_test_classification_report.json`) — should match to near-machine precision, since
   inference is deterministic; a mismatch would mean the two notebooks' pipeline definitions have drifted.
4. Computes **per-instance** SHAP values (explainer chosen by the champion's model type, same logic as Gate 4)
   on a bounded sample of test rows (same 150-row bound as Gate 4, for laptop safety regardless of which model
   family happens to be champion on a given run) — and for each sampled row, reports its top locally-important,
   provably-grounded reason codes.
5. Cross-checks this gate's own independently-computed *aggregate* top reason-code terms against Gate 4's
   already-saved global top-10 (`gate4_shap_top_features.csv`) — a real overlap count, not an assumption that
   the two SHAP runs agree.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: every number below is computed live during your run.
- **Continue gracefully on failure** (Section 17.7): SHAP is wrapped in try/except, matching Gate 4.
- **Idempotent**: re-running overwrites this gate's artifacts and appends/replaces a `gate5_...` block in
  `configs/bp1_customer_intent_classification.yaml`, without touching Gates 1-4's own fields.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp1_customer_intent_classification/artifacts/gate5_decision_records.csv` (one row per held-out
  test example — predicted intent, confidence, top-3 alternatives, reason codes where computed)
- `notebooks/bp1_customer_intent_classification/artifacts/gate5_decision_layer_summary.json`
- `notebooks/bp1_customer_intent_classification/artifacts/model_inventory_entry.json` (Gate 5 fields added)
- `configs/bp1_customer_intent_classification.yaml` — `gate5_decision_layer` block appended/updated

## Prerequisites
BP1 Gate 4 must have been real-run at least once (this notebook reads the champion from
`gate4_statistical_validation.json` and raises if that file is missing). Same `shap` requirement as Gate 4.

## If a structural check below fails
It raises `AssertionError` naming the failing check. Do not silence it.


In [ ]:
\
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp1_customer_intent_classification/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import assert_within_ram_ceiling, configure_performance, load_resource_limits  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.util  # noqa: E402
import json  # noqa: E402
import re  # noqa: E402
import time  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import yaml  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import accuracy_score  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import FunctionTransformer, LabelEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402
from catboost import CatBoostClassifier  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

if importlib.util.find_spec("shap") is None:
    raise ImportError(
        "[CHECK FAILED] The 'shap' package is required for BP1 Gate 5 and is not installed. "
        "Run `pip install shap` (inside this project's own environment) before running this notebook."
    )
import shap  # noqa: E402

print(f"[OK] shap {shap.__version__} confirmed installed (live check, not assumed).")

# ============================================================
# SECTION 4: Load Gate 3/4's real results - champion read LIVE, never hardcoded
# ============================================================
bp1_config_path = CONFIGS_DIR / "bp1_customer_intent_classification.yaml"
with open(bp1_config_path, "r", encoding="utf-8") as f:
    bp1_config = yaml.safe_load(f)
target_def = bp1_config.get("target_definition")
assert target_def is not None, "[CHECK FAILED] target_definition is still null - run BP1 Gate 1 first."
gate3_block = bp1_config.get("gate3_model_benchmark")
assert gate3_block is not None, (
    "[CHECK FAILED] gate3_model_benchmark is missing from configs/bp1_customer_intent_classification.yaml - "
    "run BP1 Gate 3 first."
)
PRIMARY_TARGET = target_def["primary_target"]
FEATURE_COL = target_def["feature_variable"]

gate4_json_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
assert gate4_json_path.exists(), (
    f"[CHECK FAILED] {gate4_json_path} not found - run BP1 Gate 4 (..._g4_statistical_validation_explainability.ipynb) "
    "first (this gate reads its confirmed champion)."
)
with open(gate4_json_path, "r", encoding="utf-8") as f:
    gate4_results = json.load(f)
CHAMPION_NAME = gate4_results["champion_model"]
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: Gate 4 recorded '{CHAMPION_NAME}' but Gate 3's config block says "
    f"'{gate3_block['champion_model']}' - these must agree; re-run Gate 3/4."
)
gate3_recorded_test_accuracy = float(gate3_block["held_out_test_accuracy"])
print(f"[OK] Champion (live, re-verified against Gate 3 + Gate 4): {CHAMPION_NAME}")

hw_summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(hw_summary_path, "r", encoding="utf-8") as f:
    hw_summary = json.load(f)
N_JOBS = hw_summary["recommended_configuration"]["recommended_n_jobs"]
cv_settings = RESOURCE_LIMITS["cv"]
RNG = np.random.RandomState(cv_settings["random_state"])

# ============================================================
# SECTION 5: Load BANKING77 train/test - identical loading to Gates 3/4
# ============================================================
B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL_DIR / "banking77_test.csv"
train_df = pd.read_csv(B77_TRAIN_PATH, dtype={"text": str, "category": str})
test_df = pd.read_csv(B77_TEST_PATH, dtype={"text": str, "category": str})

overlap = set(train_df[FEATURE_COL]) & set(test_df[FEATURE_COL])
assert len(overlap) == 0, f"[CHECK FAILED] {len(overlap)} exact-text rows overlap train/test - leakage risk."

label_encoder = LabelEncoder().fit(train_df[PRIMARY_TARGET])
X_train, y_train = train_df[FEATURE_COL], label_encoder.transform(train_df[PRIMARY_TARGET])
X_test, y_test_labels = test_df[FEATURE_COL], test_df[PRIMARY_TARGET]
y_test = label_encoder.transform(y_test_labels)
N_CLASSES = len(label_encoder.classes_)
CLASS_NAMES = label_encoder.classes_
print(f"[OK] Re-loaded BANKING77 (train={len(X_train):,}, test={len(X_test):,}, {N_CLASSES} classes).")

# ============================================================
# SECTION 6: Candidate/pipeline definitions - MUST mirror Gates 3/4's (single-source-of-truth risk, guarded
# by the live accuracy-consistency check in Section 8 below).
# ============================================================
TFIDF_KWARGS = dict(max_features=5000, ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words="english")

CANDIDATES = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=cv_settings["random_state"]),
    "random_forest": RandomForestClassifier(
        n_estimators=100, max_depth=20, n_jobs=1, random_state=cv_settings["random_state"]
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    "xgboost": XGBClassifier(
        n_estimators=100, max_depth=6, n_jobs=1, verbosity=0, random_state=cv_settings["random_state"]
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100, n_jobs=1, verbose=-1, random_state=cv_settings["random_state"]
    ),
    "catboost": CatBoostClassifier(
        iterations=100, thread_count=1, verbose=False, allow_writing_files=False,
        random_state=cv_settings["random_state"],
    ),
}
NEEDS_DENSE = {"hist_gradient_boosting"}


def _to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x


def _make_pipeline(name):
    steps = [("tfidf", TfidfVectorizer(**TFIDF_KWARGS))]
    if name in NEEDS_DENSE:
        steps.append(("densify", FunctionTransformer(_to_dense, accept_sparse=True)))
    steps.append(("clf", CANDIDATES[name]))
    return Pipeline(steps)


# ============================================================
# SECTION 7: Refit champion on FULL train, predict on the FULL held-out test set
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
champion_pipeline = _make_pipeline(CHAMPION_NAME)
print(f"\n[GATE5] Refitting champion ({CHAMPION_NAME}) on the full train split...")
t0 = time.perf_counter()
champion_pipeline.fit(X_train, y_train)
print(f"[GATE5] Fit done in {time.perf_counter() - t0:.1f}s")

assert hasattr(champion_pipeline, "predict_proba"), (
    f"[CHECK FAILED] Champion {CHAMPION_NAME} has no predict_proba - Gate 5's confidence score requires it."
)
y_pred_encoded = champion_pipeline.predict(X_test)
y_proba = champion_pipeline.predict_proba(X_test)
pipeline_classes = list(champion_pipeline.classes_)
assert pipeline_classes == list(range(N_CLASSES)), (
    f"[CHECK FAILED] Champion pipeline's class order does not match the expected 0..{N_CLASSES - 1} "
    "integer-encoded order - probability-column alignment would be wrong."
)

proba_row_sums = y_proba.sum(axis=1)
assert np.allclose(proba_row_sums, 1.0, atol=1e-6), (
    "[CHECK FAILED] predict_proba rows do not sum to 1.0 - something is wrong with the champion's probability output."
)

overall_accuracy = float(accuracy_score(y_test, y_pred_encoded))
accuracy_consistency_diff = abs(overall_accuracy - gate3_recorded_test_accuracy)
print(f"[CHECK] Recomputed overall test accuracy: {overall_accuracy:.6f} "
      f"(Gate 3 recorded: {gate3_recorded_test_accuracy:.6f}, diff={accuracy_consistency_diff:.6f})")

# Top-3 predictions per row (predicted intent + its 2 runner-up alternatives, all real predict_proba values).
top3_idx = np.argsort(y_proba, axis=1)[:, ::-1][:, :3]
top1_conf = y_proba[np.arange(len(y_proba)), top3_idx[:, 0]]
rank2_label_idx = top3_idx[:, 1]
rank2_conf = y_proba[np.arange(len(y_proba)), rank2_label_idx]
rank3_label_idx = top3_idx[:, 2]
rank3_conf = y_proba[np.arange(len(y_proba)), rank3_label_idx]

mean_conf_correct = float(top1_conf[y_pred_encoded == y_test].mean())
mean_conf_incorrect = float(top1_conf[y_pred_encoded != y_test].mean()) if (y_pred_encoded != y_test).any() else None
print(f"[RESULT] Mean top-1 confidence - correct predictions: {mean_conf_correct:.4f}, "
      f"incorrect predictions: {mean_conf_incorrect if mean_conf_incorrect is None else round(mean_conf_incorrect, 4)}")

# ============================================================
# SECTION 8: Per-instance SHAP - explainer chosen by champion's model type (mirrors Gate 4's logic), bounded
# sample for laptop safety, reason codes grounded by construction (nonzero TF-IDF weight in that exact row).
# ============================================================
SHAP_SAMPLE_SIZE = min(150, len(X_test))
N_REASON_CODES = 5
shap_error = None
sample_idx = np.array([], dtype=int)
reason_codes_by_row = {}
grounding_failures = 0
try:
    tfidf = champion_pipeline.named_steps["tfidf"]
    clf = champion_pipeline.named_steps["clf"]
    feature_names = np.array(tfidf.get_feature_names_out())

    shap_rng = np.random.RandomState(cv_settings["random_state"])
    sample_idx = shap_rng.choice(len(X_test), size=SHAP_SAMPLE_SIZE, replace=False)
    bg_idx = shap_rng.choice(len(X_train), size=min(50, len(X_train)), replace=False)
    X_sample_vec = tfidf.transform(X_test.iloc[sample_idx])
    X_bg_vec = tfidf.transform(X_train.iloc[bg_idx])
    X_sample_dense_for_masking = X_sample_vec.toarray()  # needed to mask reason codes to nonzero-weight terms

    if CHAMPION_NAME in NEEDS_DENSE:
        X_sample_vec_for_explainer = _to_dense(X_sample_vec)
        X_bg_vec_for_explainer = _to_dense(X_bg_vec)
    else:
        X_sample_vec_for_explainer = X_sample_vec
        X_bg_vec_for_explainer = X_bg_vec

    if isinstance(clf, LogisticRegression):
        print(f"\n[GATE5] SHAP: using LinearExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, background={X_bg_vec_for_explainer.shape[0]} rows)...")
        explainer = shap.LinearExplainer(clf, X_bg_vec_for_explainer)
        shap_values = explainer.shap_values(X_sample_vec_for_explainer)
    else:
        print(f"\n[GATE5] SHAP: using TreeExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, densified for SHAP's own dense-input requirement)...")
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(_to_dense(X_sample_vec))

    # Normalize to (n_samples, n_features), picking out the SHAP array for each row's OWN predicted class
    # (a per-instance explanation should explain THAT row's predicted label, not a class-averaged magnitude
    # as Gate 4's global report used).
    sample_pred_idx = y_pred_encoded[sample_idx]
    if isinstance(shap_values, list):
        # list of (n_samples, n_features) arrays, one per class
        per_row_shap = np.stack(
            [np.asarray(shap_values[cls])[i] for i, cls in enumerate(sample_pred_idx)], axis=0
        )
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            # (n_samples, n_features, n_classes) or (n_samples, n_classes, n_features) - shap's LinearExplainer
            # for a multiclass linear model returns (n_samples, n_features, n_classes) in this environment.
            if arr.shape[-1] == N_CLASSES:
                per_row_shap = arr[np.arange(len(sample_idx)), :, sample_pred_idx]
            else:
                per_row_shap = arr[np.arange(len(sample_idx)), sample_pred_idx, :]
        else:
            per_row_shap = arr

    assert per_row_shap.shape == (len(sample_idx), len(feature_names)), (
        f"[CHECK FAILED] Per-row SHAP shape {per_row_shap.shape} does not match "
        f"(sample_size={len(sample_idx)}, vocab_size={len(feature_names)})."
    )

    for local_i, global_row in enumerate(sample_idx):
        row_shap = per_row_shap[local_i]
        row_nonzero_mask = X_sample_dense_for_masking[local_i] != 0.0
        # Reason codes are ONLY ever drawn from features with nonzero TF-IDF weight in THIS row - i.e. terms
        # that are, by construction, literally present in this row's own text. This is the grounding guarantee,
        # verified explicitly below (Section 10) rather than merely assumed.
        masked_shap = np.where(row_nonzero_mask, np.abs(row_shap), -np.inf)
        if not row_nonzero_mask.any():
            reason_codes_by_row[global_row] = []
            continue
        top_k = min(N_REASON_CODES, int(row_nonzero_mask.sum()))
        top_feat_idx = np.argsort(masked_shap)[::-1][:top_k]
        codes = [str(feature_names[j]) for j in top_feat_idx]
        for j in top_feat_idx:
            if not row_nonzero_mask[j]:
                grounding_failures += 1
        reason_codes_by_row[global_row] = codes

    # Aggregate top reason-code terms across the sample (for the cross-check against Gate 4's global list).
    all_codes = [c for codes in reason_codes_by_row.values() for c in codes]
    gate5_top_terms = pd.Series(all_codes).value_counts().head(10).index.tolist() if all_codes else []
    print(f"[RESULT] Gate 5's own independently-aggregated top reason-code terms (by frequency, sampled): "
          f"{gate5_top_terms}")

except Exception as e:  # noqa: BLE001 - continue gracefully (Section 17.7)
    shap_error = f"{type(e).__name__}: {e}"
    gate5_top_terms = []
    print(f"[LIMITATION] Per-instance SHAP failed for champion model family '{type(CANDIDATES[CHAMPION_NAME]).__name__}': "
          f"{shap_error}. Decision records below will still be written with predicted labels/confidence, "
          "but with empty reason_codes.")

# ============================================================
# SECTION 9: Cross-check against Gate 4's already-saved global top-10 SHAP features
# ============================================================
gate4_shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
gate4_top_terms = []
if gate4_shap_csv_path.exists():
    gate4_shap_df = pd.read_csv(gate4_shap_csv_path)
    if len(gate4_shap_df) > 0:
        gate4_top_terms = gate4_shap_df["feature"].head(10).tolist()
overlap_terms = sorted(set(gate5_top_terms) & set(gate4_top_terms))
print(f"\n[CHECK] Gate 4 global top-10 vs Gate 5 sample-aggregated top-10 overlap: {len(overlap_terms)} terms "
      f"in common ({overlap_terms})")

# ============================================================
# SECTION 10: Assemble decision records for the FULL held-out test set (one row per test example)
# ============================================================
in_sample_set = set(int(i) for i in sample_idx)
records = []
for i in range(len(X_test)):
    codes = reason_codes_by_row.get(i, None) if i in in_sample_set else None
    records.append({
        "row_index": i,
        "true_label": str(y_test_labels.iloc[i]),
        "predicted_label": str(CLASS_NAMES[y_pred_encoded[i]]),
        "correct": bool(y_pred_encoded[i] == y_test[i]),
        "confidence_top1": round(float(top1_conf[i]), 4),
        "rank2_label": str(CLASS_NAMES[rank2_label_idx[i]]),
        "rank2_confidence": round(float(rank2_conf[i]), 4),
        "rank3_label": str(CLASS_NAMES[rank3_label_idx[i]]),
        "rank3_confidence": round(float(rank3_conf[i]), 4),
        "in_shap_sample": i in in_sample_set,
        "reason_codes": "|".join(codes) if codes else "",
        "text": str(X_test.iloc[i]),
    })
decision_records_df = pd.DataFrame(records)
assert len(decision_records_df) == len(X_test), (
    f"[CHECK FAILED] Decision-record count ({len(decision_records_df)}) does not match test-set size ({len(X_test)})."
)

records_path = ARTIFACTS_DIR / "gate5_decision_records.csv"
decision_records_df.to_csv(records_path, index=False)
print(f"\n[SAVED] {records_path.relative_to(PROJECT_ROOT)} ({len(decision_records_df):,} decision records, "
      f"{len(in_sample_set):,} with reason codes)")

# ============================================================
# SECTION 11: Write summary (idempotent overwrite-in-place)
# ============================================================
summary = {
    "bp_id": "bp1",
    "gate": 5,
    "champion_model": CHAMPION_NAME,
    "n_decision_records": int(len(decision_records_df)),
    "n_with_reason_codes": int(len(in_sample_set)),
    "shap_sample_size_bound": SHAP_SAMPLE_SIZE,
    "n_reason_codes_per_record": N_REASON_CODES,
    "shap_error": shap_error,
    "overall_test_accuracy_recomputed": round(overall_accuracy, 6),
    "gate3_recorded_test_accuracy": round(gate3_recorded_test_accuracy, 6),
    "accuracy_consistency_diff": round(accuracy_consistency_diff, 6),
    "mean_confidence_correct_predictions": round(mean_conf_correct, 4),
    "mean_confidence_incorrect_predictions": round(mean_conf_incorrect, 4) if mean_conf_incorrect is not None else None,
    "gate5_aggregated_top_reason_code_terms": gate5_top_terms,
    "gate4_global_top10_terms": gate4_top_terms,
    "overlap_terms_with_gate4": overlap_terms,
    "overlap_count_with_gate4": len(overlap_terms),
    "reason_code_grounding_failures": int(grounding_failures),
    "reason_code_grounding_method": (
        "A reason code is only ever reported for a row if that term's TF-IDF weight in that row's own "
        "vectorized text is nonzero - i.e. the term is present in that specific complaint's real text by "
        "construction of the TF-IDF vectorizer, not asserted after the fact."
    ),
    "compliance_touchpoint": {
        "udaap_language_review": (
            "Not Applicable to BP1 Gate 5 - this gate generates no GenAI or customer-facing text; reason "
            "codes and confidence scores are deterministic outputs of the champion classifier and real "
            "per-instance SHAP values. Real GenAI-drafted customer-facing text (subject to UDAAP review) is "
            "scoped to BP6 (GenAI Resolution Assistant) per the Master Execution Plan's own Section 24 "
            "Sprint 5 mapping."
        ),
        "nist_ai_rmf_measure_manage": (
            "Not Applicable to BP1 Gate 5 for the same reason - no GenAI output is produced here. Applies at BP6."
        ),
        "genai_api_used": False,
        "scope_decision_confirmed_by_user_utc": "2026-09-22",
    },
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
summary_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"[SAVED] {summary_path.relative_to(PROJECT_ROOT)}")

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
if inventory_path.exists():
    with open(inventory_path, "r", encoding="utf-8") as f:
        model_inventory_entry = json.load(f)
else:
    model_inventory_entry = {"bp_id": "bp1", "model_name": CHAMPION_NAME}
model_inventory_entry["status"] = "Gate 5 decision layer + reporting complete"
model_inventory_entry["gate5_n_decision_records"] = int(len(decision_records_df))
model_inventory_entry["gate5_overall_test_accuracy"] = round(overall_accuracy, 6)
model_inventory_entry["gate5_genai_api_used"] = False
model_inventory_entry["gate5_generated_at_utc"] = datetime.now(timezone.utc).isoformat()
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 5 fields added)")

# Order-independent patch (src/utils/bp1_config_sync.py) - replaces ONLY this gate's own
# marker-delimited block, preserving the front matter and every other gate's block regardless
# of position. See LESSONS_LEARNED_APPLIED.md #20.
from utils.bp1_config_sync import write_gate_block  # noqa: E402

new_status_value = "gate1_confirmed_gate2_confirmed_gate3_confirmed_gate4_confirmed_gate5_confirmed"
config_text = bp1_config_path.read_text(encoding="utf-8")
config_text = re.sub(r'^status:.*$', f'status: "{new_status_value}"', config_text, count=1, flags=re.MULTILINE)
bp1_config_path.write_text(config_text, encoding="utf-8")

gate5_marker = "# --- Gate 5 (Decision Layer & Reporting) results (appended, idempotent overwrite) ---"
gate5_block_lines = [
    "gate5_decision_layer:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f"  n_decision_records: {int(len(decision_records_df))}",
    f"  n_with_reason_codes: {int(len(in_sample_set))}",
    f"  overall_test_accuracy_recomputed: {round(overall_accuracy, 6)}",
    "  genai_api_used: false",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(bp1_config_path, gate5_marker, gate5_block_lines)
print(f"[SAVED] {bp1_config_path.relative_to(PROJECT_ROOT)} (gate5_decision_layer block)")

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_matches_gate3_and_gate4_recorded": CHAMPION_NAME == gate3_block["champion_model"],
    "decision_records_count_equals_test_set_size": len(decision_records_df) == len(X_test),
    "all_records_have_predicted_label_and_confidence": decision_records_df["predicted_label"].notna().all()
        and decision_records_df["confidence_top1"].notna().all(),
    "confidence_scores_within_valid_range": decision_records_df["confidence_top1"].between(0.0, 1.0).all(),
    "probabilities_sum_to_one_per_row": bool(np.allclose(proba_row_sums, 1.0, atol=1e-6)),
    "shap_sample_size_matches_configured_bound": len(sample_idx) == SHAP_SAMPLE_SIZE or shap_error is not None,
    "all_reason_codes_grounded_by_nonzero_tfidf_weight": grounding_failures == 0,
    "accuracy_consistency_with_gate3_recorded": accuracy_consistency_diff < 1e-4,
    "compliance_touchpoint_documented": "compliance_touchpoint" in summary and bool(summary["compliance_touchpoint"]),
    "decision_records_csv_written": records_path.exists(),
    "summary_json_written": summary_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp1_config_yaml_updated": bp1_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP1 Gate 5 complete. {len(decision_records_df):,} decision records written "
      f"({len(in_sample_set):,} with grounded reason codes). Recomputed test accuracy={round(overall_accuracy, 4)} "
      f"(Gate 3 recorded: {round(gate3_recorded_test_accuracy, 4)}). "
      f"Gate 4/Gate 5 SHAP top-term overlap: {len(overlap_terms)}/10. "
      "No GenAI API used (offline decision-record layer per confirmed scope decision). "
      "Proceed to BP1 Gate 6 (Productization, Monitoring & Governance) next.")
